## Introductory demos

In [51]:
BACKEND = "numba"

### README demos

In [52]:
import numpy as np

from spininfer.models.ising import IsingModel
from spininfer.data.generate import generate_data
from spininfer.data.dataset import Dataset
from spininfer.objectives.PLE_ising import PleIsingObjective
from spininfer.optimizers.adam import Adam
from spininfer.fitters.inverse_fitter import InverseFitter
from spininfer.convergence.criteria import GradientNormConvergence

model = IsingModel(n_sites=20, backend=BACKEND)

truth = generate_data(model, n_samples=20_000, iterations=1000, seed=1)
dataset = Dataset(samples=truth.samples, model=model)

fitter = InverseFitter(
    model=model,
    dataset=dataset,
    objective=PleIsingObjective(),
    optimizer=Adam(lr=0.05),
    convergence=GradientNormConvergence(model, tol=1e-6),
    n_steps=2000,
)

h_init, J_init = model.random_params(seed=2)
result = fitter.fit(h_init, J_init)

print(f"converged: {result.converged} after {result.n_steps} steps")
print(f"mean |h_err| = {np.abs(result.h - truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(result.J - truth.J).mean():.4f}")

converged: True after 289 steps
mean |h_err| = 0.0100
mean |J_err| = 0.0101


### Fitting real data
If you already have observed samples, skip `generate_data` and build a `Dataset` directly. Here the generated samples stand in for `your_data`.

In [37]:
from spininfer.models.ising import IsingModel
from spininfer.data.dataset import Dataset

your_data = np.asarray(truth.samples)

model = IsingModel(n_sites=your_data.shape[1], backend=BACKEND)
dataset = Dataset(samples=your_data, model=model)

print(dataset.n_samples, "samples,", model.n_sites, "sites")

20000 samples, 20 sites


### Using L-BFGS instead
For a lot of problems L-BFGS converges faster and with less tuning than the custom gradient loops.

In [38]:
from spininfer.fitters.lbfgs_fitter import LbfgsFitter

fitter = LbfgsFitter(model=model, dataset=dataset, objective=PleIsingObjective(), maxiter=500)
result = fitter.fit(h_init, J_init)

print(f"converged: {result.converged} after {result.n_steps} steps")
print(f"mean |h_err| = {np.abs(result.h - truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(result.J - truth.J).mean():.4f}")

converged: True after 33 steps
mean |h_err| = 0.0100
mean |J_err| = 0.0101


### Using a mean-field method instead
The mean-field methods are closed form and only need the first and second moments. They can also be used as a starting point for PLE or maximum likelihood.

In [39]:
from spininfer.fitters.mean_field_fitter import NaiveMeanFieldFitter, TAPMeanFieldFitter, IndependentPairFitter, SessakMonassonFitter, BetheFitter

mean_field_fitters = {
    "naive mean field": NaiveMeanFieldFitter(model=model, dataset=dataset),
    "TAP": TAPMeanFieldFitter(model=model, dataset=dataset),  # Ising only
    "independent pair": IndependentPairFitter(model=model, dataset=dataset),
    "Sessak-Monasson": SessakMonassonFitter(model=model, dataset=dataset),
    "Bethe": BetheFitter(model=model, dataset=dataset),  # Ising only
}

for name, mf_fitter in mean_field_fitters.items():
    mf_result = mf_fitter.fit()
    print(f"{name:>17}: mean |h_err| = {np.abs(mf_result.h - truth.h).mean():.4f}, "
          f"mean |J_err| = {np.abs(mf_result.J - truth.J).mean():.4f}")

 naive mean field: mean |h_err| = 0.1470, mean |J_err| = 0.0213
              TAP: mean |h_err| = 0.0370, mean |J_err| = 0.0199
 independent pair: mean |h_err| = 0.4062, mean |J_err| = 0.1762
  Sessak-Monasson: mean |h_err| = 0.0569, mean |J_err| = 0.0198
            Bethe: mean |h_err| = 0.0387, mean |J_err| = 0.0167


In [40]:
# Using naive mean field as a starting point for PLE
result = NaiveMeanFieldFitter(model=model, dataset=dataset).fit()
h_nmf, J_nmf = result.h, result.J

fitter = LbfgsFitter(model=model, dataset=dataset, objective=PleIsingObjective(), maxiter=500)
result = fitter.fit(h_nmf, J_nmf)

print(f"converged: {result.converged} after {result.n_steps} steps")
print(f"mean |h_err| = {np.abs(result.h - truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(result.J - truth.J).mean():.4f}")

converged: True after 28 steps
mean |h_err| = 0.0100
mean |J_err| = 0.0101


### Using adaptive cluster expansion
Adaptive cluster expansion builds the parameters from exact fits on small clusters, keeping only the clusters whose entropy contribution is larger than the threshold `theta`. Lowering `theta` keeps more clusters, which is more accurate but slower. The number of clusters grows quickly with the system size, so this demo uses a small 10-site system. Currently this method is being optimized.

In [55]:
from spininfer.fitters.ACE_fitter import ACEFitter

small_ace_model = IsingModel(n_sites=15, backend="numba")
small_ace_truth = generate_data(small_ace_model, n_samples=20_000, iterations=1000, seed=1)
small_ace_dataset = Dataset(samples=small_ace_truth.samples, model=small_ace_model)

theta = 1e-3
fitter = ACEFitter(model=small_ace_model, dataset=small_ace_dataset, threshold=theta)
ACE_result = fitter.fit()

print(f"kept clusters per size: { {k: len(v) for k, v in ACE_result.kept.items()} }")
print(f"mean |h_err| = {np.abs(ACE_result.h - small_ace_truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(ACE_result.J - small_ace_truth.J).mean():.4f}")

kept clusters per size: {1: 15, 2: 90, 3: 220, 4: 158, 5: 23, 6: 1}
mean |h_err| = 0.0319
mean |J_err| = 0.0441


### Regularization
Wrap any objective in `RegularizedObjective` to add an L1 or L2 penalty on the fields or couplings.

In [42]:
from spininfer.objectives.regularization import L2Regularizer
from spininfer.objectives.regularized import RegularizedObjective

regularized = RegularizedObjective(
    objective=PleIsingObjective(),
    regularizer=L2Regularizer(strength=0.01, apply_to_J=True),
)

result = LbfgsFitter(model=model, dataset=dataset, objective=regularized, maxiter=500).fit(h_init, J_init)
print(f"mean |h_err| = {np.abs(result.h - truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(result.J - truth.J).mean():.4f}")

mean |h_err| = 0.0092
mean |J_err| = 0.0114


### Moment matching + exact statistics (small systems)
For systems small enough to enumerate, the model moments can be computed exactly, which gives the true maximum-likelihood fit. Exact enumeration costs `2**n_sites` states per step, so this uses a smaller system. This is the most accurate method and should be used when it is possible. 

In [56]:
from spininfer.objectives.moment_matching import MomentMatchingObjective
from spininfer.stats.exact_estimator import ExactEstimator

small_model = IsingModel(n_sites=10, backend=BACKEND)
small_truth = generate_data(small_model, n_samples=20_000, iterations=1000, seed=1)
small_dataset = Dataset(samples=small_truth.samples, model=small_model)

objective = MomentMatchingObjective(estimator=ExactEstimator())
fitter = InverseFitter(model=small_model, dataset=small_dataset, objective=objective,
                       optimizer=Adam(lr=0.05), n_steps=2000)
result = fitter.fit(*small_model.random_params(seed=2))

print(f"mean |h_err| = {np.abs(result.h - small_truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(result.J - small_truth.J).mean():.4f}")

mean |h_err| = 0.0088
mean |J_err| = 0.0066


Swap `ExactEstimator()` for `McmcEstimator(n_samples=20_000, iterations=5000)` (from `spininfer.stats.mcmc_estimator`) and the same code works on systems too large to enumerate, at a much higher cost per step.

### Potts models
Same shapes, different arrays. `h` is `(n_sites, n_states)`, `J` is `(n_sites, n_sites, n_states, n_states)`, and samples are integer state labels. Everything else works as in the Ising case, with `PlePottsObjective` in place of `PleIsingObjective`.

In [58]:
from spininfer.models.potts import PottsModel
from spininfer.objectives.PLE_potts import PlePottsObjective

potts_model = PottsModel(n_sites=15, n_states=3, backend=BACKEND)
potts_truth = generate_data(potts_model, n_samples=20_000, iterations=1000, seed=1)
potts_dataset = Dataset(samples=potts_truth.samples, model=potts_model)

fitter = LbfgsFitter(model=potts_model, dataset=potts_dataset, objective=PlePottsObjective(), maxiter=500)
result = fitter.fit(*potts_model.random_params(seed=2))

print(f"converged: {result.converged} after {result.n_steps} steps")
print(f"mean |h_err| = {np.abs(result.h - potts_truth.h).mean():.4f}")
print(f"mean |J_err| = {np.abs(result.J - potts_truth.J).mean():.4f}")

converged: True after 32 steps
mean |h_err| = 0.0159
mean |J_err| = 0.0135


All other methods above for the Ising can be used identically for the Potts model.

### Thermodynamic quantities
Given the parameters, the free energy, energy, entropy and heat capacity can be computed exactly or by thermodynamic integration over MCMC samples. Units are β = 1.

In [59]:
from spininfer.models.ising import IsingModel
from spininfer.stats.exact_thermodynamics_estimator import ExactThermodynamicsEstimator
from spininfer.stats.thermodynamics_integration_estimator import ThermodynamicsIntegrationEstimator

model = IsingModel(n_sites=12, backend=BACKEND)
h, J = model.random_params(seed=1)

exact = ExactThermodynamicsEstimator().estimate(model, h, J)
ti = ThermodynamicsIntegrationEstimator(n_samples=10_000, iterations=1000).estimate(model, h, J)

print(f"{'':>8} {'free energy':>12} {'energy':>10} {'entropy':>10} {'heat cap.':>10}")
print(f"{'exact':>8} {exact.free_energy:12.4f} {exact.energy:10.4f} {exact.entropy:10.4f} {exact.heat_capacity:10.4f}")
print(f"{'TI':>8} {ti.free_energy:12.4f} {ti.energy:10.4f} {ti.entropy:10.4f} {ti.heat_capacity:10.4f}")

          free energy     energy    entropy  heat cap.
   exact     -10.7184    -5.2501     5.4684     5.5642
      TI     -10.7151    -5.2169     5.4983     5.6302
